# Re-Checking the Filler-vs-Elaboration Answer Gap

This notebook re-analyzes a 336-row `(prompt, model)` dataset from a prior GSM8K experiment that measured how **content type** (bare / filler / relevant-elaboration) and **length tier** (short / medium / long) affect the **coefficient of variation (CV)** of numeric answers across 3 OpenAI-hosted models.

Instead of trusting raw point estimates over 7 condition-mean rows, this script recomputes everything with defensible inferential statistics:

- **Metric 1**: paired relevant-minus-filler CV delta per seed, with a cluster (block) bootstrap over seed_ids and a paired Wilcoxon signed-rank test, per length tier.
- **Metric 2**: cell-level (not condition-mean) correlations between CV and two entropy proxies, with both a naive row-level bootstrap CI (flagged anti-conservative) and a seed-cluster bootstrap CI.
- **Metric 3**: a per-model x (content_type, length_tier) breakdown table.
- **Metric 4**: MAD/median and 5%-trimmed CV as robustness checks against the standard CV.
- **Metric 5**: a check (skipped here) for a newer 4-condition decomposition artifact.

This demo runs on a small curated subset (`mini_demo_data.json`, 6 seeds instead of 16) of the original data so it executes in a couple of minutes."

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, scipy, matplotlib -- pre-installed on Colab, install locally only (exact Colab versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd
from loguru import logger
from scipy import stats
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-0e9809-interpretive-load-not-token-count-drives/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['prompt_model_results'])} prompt_model rows and {len(data['raw_completions'])} raw completion rows")

## Config

All tunable parameters live here. `N_BOOT` is the number of bootstrap resamples used throughout the analysis -- the original script used 10,000; this demo uses a much smaller value so the notebook finishes quickly. Increase it (up to 10,000, the original value) for tighter CIs if you have more time."

In [ ]:
RNG_SEED = 12345
N_BOOT = 200  # original script: 10_000 -- reduced here for demo speed

## Step 0: blocker check

The original script first checks that the upstream dependency files (`full_method_out.json`, `prompt_model_results.csv`, `raw_completions.jsonl`) exist and parse before doing any analysis. Here `data` has already been loaded from `mini_demo_data.json`, so this cell just re-checks non-emptiness and parseability of the two tables it contains, matching the spirit of the original check."

In [ ]:
def jsonable(x):
    """Recursively convert numpy/pandas scalars to native python for json.dumps."""
    if isinstance(x, dict):
        return {k: jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.floating,)):
        v = float(x)
        return None if not np.isfinite(v) else v
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, float):
        return None if not np.isfinite(x) else x
    return x


# ---------------------------------------------------------------------------
# STEP 0: blocker check (adapted to check the in-memory `data` dict instead of files)
# ---------------------------------------------------------------------------
def step0_blocker_check(data) -> dict:
    logger.info("STEP 0: checking dependency data exist and are non-empty/parseable")
    pm_rows = data.get("prompt_model_results", [])
    raw_rows = data.get("raw_completions", [])

    n_rows = len(pm_rows)
    if n_rows == 0:
        return {"blocked": True, "missing_files": ["prompt_model_results (0 rows)"]}

    n_lines = len(raw_rows)
    if n_lines == 0:
        return {"blocked": True, "missing_files": ["raw_completions (0 lines)"]}

    logger.info(
        f"STEP 0 PASSED: prompt_model_results n_rows={n_rows}, "
        f"raw_completions n_lines={n_lines}"
    )
    return {
        "blocked": False,
        "n_rows_prompt_model_csv": n_rows,
        "n_lines_raw_completions": n_lines,
        "n_bad_lines_raw_completions": 0,
    }


blocker = step0_blocker_check(data)
blocker

## Step 1: load & reconcile schema

Builds a tidy per-`(prompt, model)` DataFrame from the `prompt_model_results` table (deriving `seed_id` from `prompt_id`, renaming a few columns, dropping rows with NaN CV from division-by-zero when the mean answer is 0), plus a long-format DataFrame of individual raw completions."

In [ ]:
def step1_load(data) -> tuple[pd.DataFrame, pd.DataFrame]:
    logger.info("STEP 1: loading prompt_model_results and raw_completions")
    df = pd.DataFrame(data["prompt_model_results"])

    # Derive seed_id from prompt_id (format: seed_XXX_<content_type>_<length_tier>)
    df["seed_id"] = df["prompt_id"].str.extract(r"^(seed_\d+)_")

    df = df.rename(
        columns={
            "answer_cv": "cv",
            "answer_variance": "variance",
            "mean_logprob_entropy_first_k": "mean_entropy_first_k",
        }
    )

    keep_cols = [
        "prompt_id",
        "model",
        "seed_id",
        "content_type",
        "length_tier",
        "cv",
        "variance",
        "frac_correct",
        "mean_entropy_first_k",
        "mean_answer_token_entropy",
        "n_valid_samples",
    ]
    tidy = df[keep_cols].copy()
    tidy = tidy.rename(columns={"mean_answer_token_entropy": "answer_token_entropy"})

    n_before = len(tidy)
    nan_cv_rows = tidy[tidy["cv"].isna()]
    if len(nan_cv_rows) > 0:
        logger.warning(
            f"Dropping {len(nan_cv_rows)} rows with NaN CV (division-by-zero when answer_mean=0): "
            f"{nan_cv_rows['prompt_id'].tolist()}"
        )
        tidy = tidy.dropna(subset=["cv"]).reset_index(drop=True)
    logger.info(
        f"Tidy dataframe: {len(tidy)} rows (dropped {n_before - len(tidy)} NaN-CV rows), "
        f"{tidy['seed_id'].nunique()} unique seeds"
    )

    raw_df = pd.DataFrame(
        [
            {
                "prompt_id": r.get("prompt_id"),
                "model": r.get("model"),
                "sample_idx": r.get("sample_idx"),
                "answer": r.get("answer"),
            }
            for r in data["raw_completions"]
        ]
    )
    logger.info(f"Raw completions dataframe: {len(raw_df)} rows")
    return tidy, raw_df


tidy, raw_df = step1_load(data)
tidy.head()

## Bootstrap helpers

Three reusable helpers used by every metric below: a cluster (block) bootstrap over cluster-level means, a naive row-level bootstrap for a correlation coefficient, and a seed-cluster bootstrap for a correlation coefficient (resampling whole `seed_id` groups instead of individual rows, since rows sharing a `seed_id` are not independent)."

In [ ]:
def cluster_bootstrap_mean(values_by_cluster: list[np.ndarray], n_boot: int, rng: np.random.Generator):
    """Cluster (block) bootstrap on cluster-level means: resample clusters with
    replacement, compute mean-of-cluster-means, repeat n_boot times."""
    cluster_means = np.array([np.nanmean(v) for v in values_by_cluster if len(v) > 0])
    n_clusters = len(cluster_means)
    if n_clusters == 0:
        return None
    boot_means = np.empty(n_boot)
    idx_pool = np.arange(n_clusters)
    for b in range(n_boot):
        idx = rng.choice(idx_pool, size=n_clusters, replace=True)
        boot_means[b] = np.mean(cluster_means[idx])
    return {
        "n_clusters": int(n_clusters),
        "mean": float(np.mean(cluster_means)),
        "ci_lower": float(np.percentile(boot_means, 2.5)),
        "ci_upper": float(np.percentile(boot_means, 97.5)),
    }


def bootstrap_corr(x: np.ndarray, y: np.ndarray, n_boot: int, rng: np.random.Generator, method: str):
    n = len(x)
    if n < 3:
        return None
    boot_vals = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb, yb = x[idx], y[idx]
        if np.std(xb) == 0 or np.std(yb) == 0:
            boot_vals[b] = np.nan
            continue
        if method == "pearson":
            boot_vals[b] = stats.pearsonr(xb, yb)[0]
        else:
            boot_vals[b] = stats.spearmanr(xb, yb)[0]
    boot_vals = boot_vals[~np.isnan(boot_vals)]
    if method == "pearson":
        r, p = stats.pearsonr(x, y)
    else:
        r, p = stats.spearmanr(x, y)
    return {
        "n": int(n),
        "statistic": float(r),
        "p_value": float(p),
        "ci_lower": float(np.percentile(boot_vals, 2.5)) if len(boot_vals) else None,
        "ci_upper": float(np.percentile(boot_vals, 97.5)) if len(boot_vals) else None,
    }


def cluster_bootstrap_corr(df: pd.DataFrame, xcol: str, ycol: str, n_boot: int, rng: np.random.Generator, method: str):
    """Resample seed_ids with replacement, pool all rows for the resampled seeds, recompute corr.
    Uses plain numpy arrays (not pandas concat) to avoid per-iteration allocation overhead."""
    seeds = df["seed_id"].unique()
    n_seeds = len(seeds)
    seed_to_xy = {
        s: (grp[xcol].values.astype(float), grp[ycol].values.astype(float))
        for s, grp in df.groupby("seed_id")
    }
    x_by_seed = [seed_to_xy[s][0] for s in seeds]
    y_by_seed = [seed_to_xy[s][1] for s in seeds]

    boot_vals = np.empty(n_boot)
    for b in range(n_boot):
        chosen = rng.integers(0, n_seeds, size=n_seeds)
        x = np.concatenate([x_by_seed[i] for i in chosen])
        y = np.concatenate([y_by_seed[i] for i in chosen])
        if np.std(x) == 0 or np.std(y) == 0:
            boot_vals[b] = np.nan
            continue
        if method == "pearson":
            boot_vals[b] = stats.pearsonr(x, y)[0]
        else:
            boot_vals[b] = stats.spearmanr(x, y)[0]
    boot_vals = boot_vals[~np.isnan(boot_vals)]
    return {
        "n_seeds": int(n_seeds),
        "ci_lower": float(np.percentile(boot_vals, 2.5)) if len(boot_vals) else None,
        "ci_upper": float(np.percentile(boot_vals, 97.5)) if len(boot_vals) else None,
    }

## Metric 1: paired filler-vs-elaboration CV gap

For each length tier, computes the per-seed (relevant - filler) CV delta averaged across models, a cluster bootstrap 95% CI over the seed_ids, and a paired Wilcoxon signed-rank test. Also runs the same per model, and pools across tiers using seed x tier as the cluster unit."

In [ ]:
def metric1_paired_gap(df: pd.DataFrame, rng: np.random.Generator) -> dict:
    logger.info("METRIC 1: paired filler-vs-elaboration CV gap with cluster bootstrap")
    results = {"per_tier": {}, "per_tier_per_model": {}}

    tiers = sorted(df.loc[df["content_type"].isin(["relevant", "filler"]), "length_tier"].unique())
    logger.info(f"Tiers found (excluding bare): {tiers}")

    all_pooled_deltas_by_cluster = []  # for pooled seed x tier cluster unit

    for tier in tiers:
        sub = df[(df["length_tier"] == tier) & (df["content_type"].isin(["relevant", "filler"]))]
        # per-seed, per-model paired delta, then average across models per seed
        pivot = sub.pivot_table(
            index=["seed_id", "model"], columns="content_type", values="cv", aggfunc="mean"
        ).reset_index()
        pivot = pivot.dropna(subset=["relevant", "filler"])
        pivot["delta"] = pivot["relevant"] - pivot["filler"]

        # per-seed averaged across models
        per_seed = pivot.groupby("seed_id")["delta"].mean()
        seed_ids = per_seed.index.tolist()
        deltas_by_cluster = [np.array([per_seed[s]]) for s in seed_ids]

        boot = cluster_bootstrap_mean(deltas_by_cluster, N_BOOT, rng)
        wstat, wp = stats.wilcoxon(per_seed.values, alternative="two-sided", zero_method="wilcox")

        results["per_tier"][str(tier)] = {
            "n_seeds": int(len(per_seed)),
            "mean_delta_relevant_minus_filler_cv": float(per_seed.mean()),
            "ci_95_lower": boot["ci_lower"] if boot else None,
            "ci_95_upper": boot["ci_upper"] if boot else None,
            "wilcoxon_statistic": float(wstat),
            "wilcoxon_p_value": float(wp),
            "ci_excludes_zero": bool(boot and (boot["ci_lower"] > 0 or boot["ci_upper"] < 0)),
        }

        # accumulate for pooled seed x tier cluster
        for s in seed_ids:
            all_pooled_deltas_by_cluster.append(np.array([per_seed[s]]))

        # per-model breakdown reused later in metric 3, but compute per-tier-per-model deltas here too
        results["per_tier_per_model"][str(tier)] = {}
        for model, mdf in pivot.groupby("model"):
            per_seed_m = mdf.set_index("seed_id")["delta"]
            deltas_by_cluster_m = [np.array([v]) for v in per_seed_m.values]
            boot_m = cluster_bootstrap_mean(deltas_by_cluster_m, N_BOOT, rng)
            if len(per_seed_m) >= 1 and np.any(per_seed_m.values != 0):
                try:
                    wstat_m, wp_m = stats.wilcoxon(per_seed_m.values, alternative="two-sided", zero_method="wilcox")
                except ValueError:
                    wstat_m, wp_m = np.nan, np.nan
            else:
                wstat_m, wp_m = np.nan, np.nan
            results["per_tier_per_model"][str(tier)][model] = {
                "n_seeds": int(len(per_seed_m)),
                "mean_delta": float(per_seed_m.mean()),
                "ci_95_lower": boot_m["ci_lower"] if boot_m else None,
                "ci_95_upper": boot_m["ci_upper"] if boot_m else None,
                "wilcoxon_statistic": None if np.isnan(wstat_m) else float(wstat_m),
                "wilcoxon_p_value": None if np.isnan(wp_m) else float(wp_m),
            }

    # pooled across tiers, seed x tier as cluster unit
    boot_pooled = cluster_bootstrap_mean(all_pooled_deltas_by_cluster, N_BOOT, rng)
    flat_deltas = np.array([v[0] for v in all_pooled_deltas_by_cluster])
    wstat_p, wp_p = stats.wilcoxon(flat_deltas, alternative="two-sided", zero_method="wilcox")
    results["pooled_across_tiers_seed_x_tier_cluster"] = {
        "n_clusters": int(len(all_pooled_deltas_by_cluster)),
        "mean_delta": float(flat_deltas.mean()),
        "ci_95_lower": boot_pooled["ci_lower"] if boot_pooled else None,
        "ci_95_upper": boot_pooled["ci_upper"] if boot_pooled else None,
        "wilcoxon_statistic": float(wstat_p),
        "wilcoxon_p_value": float(wp_p),
        "ci_excludes_zero": bool(boot_pooled and (boot_pooled["ci_lower"] > 0 or boot_pooled["ci_upper"] < 0)),
    }
    return results


rng = np.random.default_rng(RNG_SEED)
m1 = metric1_paired_gap(tidy, rng)
m1["per_tier"]

## Metric 2: cell-level entropy-CV correlation

Pearson and Spearman correlations between CV and two entropy proxies (`mean_entropy_first_k`, `answer_token_entropy`), computed over ALL `(prompt, model)` cells (not condition means), with both a naive row-level bootstrap CI (flagged anti-conservative since rows share `seed_id`) and a seed-cluster bootstrap CI. Also recomputed within each `content_type` subset to test whether entropy tracks CV beyond just condition membership."

In [ ]:
def metric2_correlations(df: pd.DataFrame, rng: np.random.Generator) -> dict:
    logger.info("METRIC 2: cell-level entropy-CV correlation with bootstrap CI")
    out = {"all_rows": {}, "by_content_type": {}}

    pairs = [
        ("cv", "mean_entropy_first_k"),
        ("cv", "answer_token_entropy"),
    ]

    for xcol, ycol in pairs:
        x = df[xcol].values.astype(float)
        y = df[ycol].values.astype(float)
        key = f"{xcol}_vs_{ycol}"
        out["all_rows"][key] = {}
        for method in ("pearson", "spearman"):
            naive = bootstrap_corr(x, y, N_BOOT, rng, method)
            cluster = cluster_bootstrap_corr(df, xcol, ycol, N_BOOT, rng, method)
            out["all_rows"][key][method] = {
                **naive,
                "cluster_bootstrap_ci_95_lower": cluster["ci_lower"],
                "cluster_bootstrap_ci_95_upper": cluster["ci_upper"],
                "cluster_bootstrap_n_seeds": cluster["n_seeds"],
                "note": "naive row-level bootstrap likely anti-conservative: rows share seed_id and are not fully independent",
            }

    for ct in df["content_type"].unique():
        sub = df[df["content_type"] == ct]
        out["by_content_type"][ct] = {}
        for xcol, ycol in pairs:
            x = sub[xcol].values.astype(float)
            y = sub[ycol].values.astype(float)
            key = f"{xcol}_vs_{ycol}"
            out["by_content_type"][ct][key] = {}
            for method in ("pearson", "spearman"):
                res = bootstrap_corr(x, y, N_BOOT, rng, method)
                out["by_content_type"][ct][key][method] = res
    return out


m2 = metric2_correlations(tidy, rng)
m2["all_rows"]["cv_vs_mean_entropy_first_k"]["pearson"]

## Metric 3: per-model x condition breakdown

A table of mean CV, both entropy proxies, `frac_correct`, and `n` for each `(model, content_type, length_tier)` cell, used to check whether the pooled pattern is driven by one model."

In [ ]:
def metric3_per_model_breakdown(df: pd.DataFrame) -> dict:
    logger.info("METRIC 3: per-model x condition breakdown table")
    table = {}
    for model, mdf in df.groupby("model"):
        table[model] = {}
        for (ct, lt), cell in mdf.groupby(["content_type", "length_tier"]):
            key = f"{ct}|{lt}"
            table[model][key] = {
                "n": int(len(cell)),
                "mean_cv": float(cell["cv"].mean()),
                "mean_entropy_first_k": float(cell["mean_entropy_first_k"].mean()),
                "mean_answer_token_entropy": float(cell["answer_token_entropy"].mean()),
                "mean_frac_correct": float(cell["frac_correct"].mean()),
            }
    return table


m3 = metric3_per_model_breakdown(tidy)
list(m3.keys())

## Metric 4: robust/outlier-trimmed dispersion

Computes MAD/median and a 5%-trimmed CV per `(prompt, model)` cell from the raw per-sample answers (flagging cells with `n_valid_samples < 10` as too-small-to-trim), then re-runs the Metric 1 cluster-bootstrap gap using MAD and trimmed-CV in place of standard CV, to check the gap's robustness to outliers."

In [ ]:
def metric4_robust_dispersion(df: pd.DataFrame, raw_df: pd.DataFrame, rng: np.random.Generator) -> dict:
    logger.info("METRIC 4: robust/outlier-trimmed dispersion")
    cell_stats = []
    too_small = []
    for (pid, model), grp in raw_df.groupby(["prompt_id", "model"]):
        vals = grp["answer"].dropna().values.astype(float)
        n = len(vals)
        if n == 0:
            continue
        median = np.median(vals)
        mad = np.median(np.abs(vals - median))
        mad_over_median = (mad / abs(median)) if median != 0 else np.nan

        if n < 10:
            too_small.append({"prompt_id": pid, "model": model, "n_valid_samples": int(n)})
            trimmed_cv = np.nan
        else:
            lo, hi = np.percentile(vals, [5, 95])
            trimmed_vals = vals[(vals >= lo) & (vals <= hi)]
            if len(trimmed_vals) >= 2 and np.mean(trimmed_vals) != 0:
                trimmed_cv = np.std(trimmed_vals, ddof=1) / abs(np.mean(trimmed_vals))
            else:
                trimmed_cv = np.nan

        cell_stats.append(
            {
                "prompt_id": pid,
                "model": model,
                "n_valid_samples": int(n),
                "mad_over_median": mad_over_median,
                "trimmed_cv": trimmed_cv,
            }
        )

    cell_df = pd.DataFrame(cell_stats)
    merged = df.merge(cell_df, on=["prompt_id", "model"], how="left")

    tiers = sorted(merged.loc[merged["content_type"].isin(["relevant", "filler"]), "length_tier"].unique())
    out = {"too_small_to_trim_n_cells": len(too_small), "too_small_cells": too_small[:50], "per_tier": {}}

    for tier in tiers:
        sub = merged[(merged["length_tier"] == tier) & (merged["content_type"].isin(["relevant", "filler"]))]
        tier_res = {}
        for metric_col, label in [("cv", "standard_cv"), ("mad_over_median", "mad_over_median"), ("trimmed_cv", "trimmed_cv")]:
            pivot = sub.pivot_table(
                index=["seed_id", "model"], columns="content_type", values=metric_col, aggfunc="mean"
            ).reset_index()
            pivot = pivot.dropna(subset=["relevant", "filler"])
            if len(pivot) == 0:
                tier_res[label] = None
                continue
            pivot["delta"] = pivot["relevant"] - pivot["filler"]
            per_seed = pivot.groupby("seed_id")["delta"].mean()
            deltas_by_cluster = [np.array([v]) for v in per_seed.values]
            boot = cluster_bootstrap_mean(deltas_by_cluster, N_BOOT, rng)
            tier_res[label] = {
                "n_seeds": int(len(per_seed)),
                "mean_delta": float(per_seed.mean()),
                "ci_95_lower": boot["ci_lower"] if boot else None,
                "ci_95_upper": boot["ci_upper"] if boot else None,
            }
        out["per_tier"][str(tier)] = tier_res

    return out


m4 = metric4_robust_dispersion(tidy, raw_df, rng)
m4["per_tier"]

## Metric 5 (conditional, skipped here)

The original script checks the run's artifact tree for a newer 4-condition decomposition artifact (paraphrase-only vs paraphrase+scaffolding vs original elaboration vs filler). That check depends on scanning other artifacts in the full pipeline run directory, which isn't part of this standalone demo, so it's represented here as a static skip result matching what the original run found."

In [ ]:
m5 = {"skipped": True, "reason": "No additional decomposition experiment/dataset artifacts found in the run's artifact directory beyond the dependency already analyzed."}
m5

## Narrative verdicts and superseded numbers

Turns the raw metric outputs into an explicit STATISTICALLY_SUPPORTED / REMAINS_DESCRIPTIVE / NOT_SUPPORTED verdict per hypothesis claim, and lists the prior draft's numbers that must stop being cited in favor of these CI-qualified figures."

In [ ]:
def build_narrative(m1: dict, m2: dict, m4: dict) -> dict:
    verdicts = {}

    # Claim A: elaboration destabilizes more than filler at every tier
    supported_tiers = []
    remains_descriptive_tiers = []
    for tier, res in m1["per_tier"].items():
        if res["ci_excludes_zero"] and res["mean_delta_relevant_minus_filler_cv"] > 0:
            supported_tiers.append(tier)
        else:
            remains_descriptive_tiers.append(tier)

    claim_a_status = (
        "STATISTICALLY_SUPPORTED" if len(remains_descriptive_tiers) == 0
        else ("REMAINS_DESCRIPTIVE" if len(supported_tiers) > 0 else "NOT_SUPPORTED")
    )
    verdicts["claim_elaboration_destabilizes_more_than_filler"] = {
        "status": claim_a_status,
        "tiers_ci_excludes_zero_and_positive": supported_tiers,
        "tiers_ci_crosses_zero_or_negative": remains_descriptive_tiers,
        "pooled_ci_excludes_zero": m1["pooled_across_tiers_seed_x_tier_cluster"]["ci_excludes_zero"],
    }

    # Claim B: entropy correlates with / mediates CV
    ent_first_k = m2["all_rows"]["cv_vs_mean_entropy_first_k"]["pearson"]
    ent_token = m2["all_rows"]["cv_vs_answer_token_entropy"]["pearson"]

    def corr_supported(res):
        return res["cluster_bootstrap_ci_95_lower"] is not None and (
            res["cluster_bootstrap_ci_95_lower"] > 0 or res["cluster_bootstrap_ci_95_upper"] < 0
        )

    within_condition_signal = any(
        m2["by_content_type"][ct]["cv_vs_mean_entropy_first_k"]["pearson"] is not None
        and m2["by_content_type"][ct]["cv_vs_mean_entropy_first_k"]["pearson"]["ci_lower"] is not None
        and (
            m2["by_content_type"][ct]["cv_vs_mean_entropy_first_k"]["pearson"]["ci_lower"] > 0
            or m2["by_content_type"][ct]["cv_vs_mean_entropy_first_k"]["pearson"]["ci_upper"] < 0
        )
        for ct in m2["by_content_type"]
    )

    claim_b_status = "STATISTICALLY_SUPPORTED" if (corr_supported(ent_first_k) or corr_supported(ent_token)) else "REMAINS_DESCRIPTIVE"
    verdicts["claim_entropy_correlates_with_cv"] = {
        "status": claim_b_status,
        "cell_level_pearson_r_cv_vs_mean_entropy_first_k": ent_first_k["statistic"],
        "cell_level_pearson_r_cv_vs_answer_token_entropy": ent_token["statistic"],
        "cluster_bootstrap_ci_excludes_zero_first_k": corr_supported(ent_first_k),
        "cluster_bootstrap_ci_excludes_zero_token": corr_supported(ent_token),
        "within_content_type_signal_survives": within_condition_signal,
        "interpretation": (
            "Correlation partly driven by between-condition variance rather than a true within-condition "
            "relationship" if not within_condition_signal else
            "Some within-condition signal survives, weakening the pure between-condition-variance explanation"
        ),
    }

    # Claim C: robustness to outliers
    robust_holds = []
    for tier, res in m4["per_tier"].items():
        std_d = res.get("standard_cv")
        mad_d = res.get("mad_over_median")
        trim_d = res.get("trimmed_cv")
        if std_d and mad_d and trim_d:
            same_sign = (std_d["mean_delta"] > 0) == (mad_d["mean_delta"] > 0) == (trim_d["mean_delta"] > 0)
            robust_holds.append(same_sign)
    verdicts["claim_gap_robust_to_outliers"] = {
        "status": "STATISTICALLY_SUPPORTED" if robust_holds and all(robust_holds) else "REMAINS_DESCRIPTIVE",
        "tiers_checked": len(robust_holds),
        "tiers_same_sign_across_cv_mad_trimmed": sum(robust_holds),
    }

    return verdicts


def superseded_numbers(m1: dict, m2: dict) -> list[str]:
    items = [
        "The prior draft's r=0.75/r=0.59 entropy-CV correlations computed over 7 condition-mean rows are SUPERSEDED by "
        f"cell-level correlations: Pearson r(cv, mean_entropy_first_k)={m2['all_rows']['cv_vs_mean_entropy_first_k']['pearson']['statistic']:.3f}, "
        f"Pearson r(cv, answer_token_entropy)={m2['all_rows']['cv_vs_answer_token_entropy']['pearson']['statistic']:.3f}. "
        "Downstream text must cite the cell-level r/rho with bootstrap CIs, not the condition-mean r.",
        "The raw CV point estimates by content_type x length_tier in the prior draft (e.g. filler medium 0.277, relevant medium 0.474) "
        "are SUPERSEDED as evidence of a 'gap' by the paired, seed-clustered bootstrap deltas and Wilcoxon tests in Metric 1 -- "
        "the point estimates themselves are retained as descriptive means but must be reported alongside the CI/p-value, never alone.",
        "Any claim that the elaboration>filler pattern is general is SUPERSEDED by the per-model breakdown (Metric 3): the pattern "
        "must be checked/reported per model, since all 3 models are same-provider/same-family.",
        "Any claim about CV-based gap magnitude that does not address outlier sensitivity is SUPERSEDED by the MAD/trimmed-CV cross-check (Metric 4).",
    ]
    return items


narrative = build_narrative(m1, m2, m4)
superseded = superseded_numbers(m1, m2)
narrative

## Results

Print the key numbers in a readable table and plot the per-tier CV gap (with bootstrap CIs) and the cell-level entropy-CV correlations."

In [ ]:
print("=== Metric 1: paired relevant-minus-filler CV gap per tier ===")
for tier, res in m1["per_tier"].items():
    print(
        f"  {tier:>7s}: mean_delta={res['mean_delta_relevant_minus_filler_cv']:+.3f}  "
        f"95% CI=[{res['ci_95_lower']:+.3f}, {res['ci_95_upper']:+.3f}]  "
        f"wilcoxon_p={res['wilcoxon_p_value']:.3f}  ci_excludes_zero={res['ci_excludes_zero']}"
    )

print("\n=== Metric 2: cell-level entropy-CV correlations (Pearson) ===")
for key in ["cv_vs_mean_entropy_first_k", "cv_vs_answer_token_entropy"]:
    r = m2["all_rows"][key]["pearson"]
    print(
        f"  {key}: r={r['statistic']:+.3f} (p={r['p_value']:.3g}), "
        f"cluster-bootstrap 95% CI=[{r['cluster_bootstrap_ci_95_lower']:+.3f}, {r['cluster_bootstrap_ci_95_upper']:+.3f}]"
    )

print("\n=== Narrative verdicts ===")
for claim, v in narrative.items():
    print(f"  {claim}: {v['status']}")

# --- Plot: per-tier CV gap with bootstrap CIs ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

tiers = list(m1["per_tier"].keys())
means = [m1["per_tier"][t]["mean_delta_relevant_minus_filler_cv"] for t in tiers]
lo = [m1["per_tier"][t]["ci_95_lower"] for t in tiers]
hi = [m1["per_tier"][t]["ci_95_upper"] for t in tiers]
err = [[m - l for m, l in zip(means, lo)], [h - m for m, h in zip(means, hi)]]

ax = axes[0]
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.errorbar(tiers, means, yerr=err, fmt="o", capsize=5, color="tab:blue")
ax.set_title("Relevant - Filler CV gap by length tier")
ax.set_ylabel("mean CV delta (relevant - filler)")

ax = axes[1]
x = tidy["mean_entropy_first_k"].values.astype(float)
y = tidy["cv"].values.astype(float)
ax.scatter(x, y, alpha=0.6, s=18, color="tab:orange")
r = m2["all_rows"]["cv_vs_mean_entropy_first_k"]["pearson"]["statistic"]
ax.set_title(f"CV vs mean_entropy_first_k (cell-level r={r:.3f})")
ax.set_xlabel("mean_entropy_first_k")
ax.set_ylabel("CV")

plt.tight_layout()
plt.show()